# TF-IDF + LinearSVC Nâng Cấp v1 — Phân Loại Cảm Xúc Tiếng Việt

Phiên bản nâng cấp với các cải tiến kỹ thuật:

- **FeatureUnion**: kết hợp word n-gram (1–3) và char n-gram (3–5) với sublinear TF-IDF
- **SelectKBest(chi2)**: lọc đặc trưng có ý nghĩa thống kê cao nhất
- **LinearSVC** + `class_weight='balanced'` để xử lý mất cân bằng lớp
- **CalibratedClassifierCV(sigmoid)** để cải thiện xác suất và F1
- **RandomizedSearchCV** với **StratifiedKFold(k=5)** để tìm siêu tham số tối ưu

**Mục tiêu:** Accuracy và F1-macro tối đa (hướng đến > 95% nếu dữ liệu đủ tốt).


In [ ]:
# Cài đặt thư viện tương thích Google Colab Python 3.12
!pip -q install --upgrade \
    "numpy==2.0.2" "scipy==1.13.1" "pandas==2.2.2" \
    "scikit-learn==1.6.1" "matplotlib==3.8.4" "seaborn==0.13.2" "joblib==1.4.2"

print("Cài đặt xong.")


## 1. Imports và Cấu hình

In [ ]:
import os
import re
import json
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from google.colab import drive
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, RandomizedSearchCV
)
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings("ignore")

# Seed cố định cho tính tái tạo (reproducibility)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Thử import seaborn; nếu lỗi thì dùng matplotlib thuần
try:
    import seaborn as sns
    HAS_SEABORN = True
except Exception:
    sns = None
    HAS_SEABORN = False
    print("Canh bao: Khong import duoc seaborn, dung matplotlib.")

# Nhãn lớp
LABEL_MAP  = {0: "Tieu cuc", 1: "Tich cuc", 2: "Trung tinh"}
CLASS_NAMES = [LABEL_MAP[i] for i in sorted(LABEL_MAP)]

print("Thu vien da san sang.")
print(f"HAS_SEABORN = {HAS_SEABORN}")


## 2. Kết nối Google Drive và Đường dẫn

In [ ]:
drive.mount("/content/drive")

BASE_DIR  = "/content/drive/MyDrive"
DATA_PATH = f"{BASE_DIR}/data/data_v2.1/data_v2.1.csv"
OUT_DIR   = f"{BASE_DIR}/tfidf-svm-v1"
os.makedirs(OUT_DIR, exist_ok=True)

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"Khong tim thay file: {DATA_PATH}")

print(f"DATA_PATH ton tai : {os.path.exists(DATA_PATH)}")
print(f"OUT_DIR           : {OUT_DIR}")


## 3. Tải và Làm sạch Dữ liệu

**Chiến lược làm sạch văn bản tiếng Việt:**
- Giữ nguyên dấu (diacritics) — bắt buộc với tiếng Việt.
- Xóa URL, HTML tags, emoji.
- Chuẩn hóa khoảng trắng, chuyển về chữ thường.
- Không dùng `strip_accents` trong TF-IDF để bảo toàn ngữ nghĩa.


In [ ]:
# Pattern xoa emoji Unicode
_EMOJI_RE = re.compile(
    r"["
    r"\U0001F600-\U0001F64F"
    r"\U0001F300-\U0001F5FF"
    r"\U0001F680-\U0001F6FF"
    r"\U0001F1E0-\U0001F1FF"
    r"\U00010000-\U0010FFFF"
    r"]+",
    flags=re.UNICODE,
)

def clean_text(s: str) -> str:
    """Lam sach van ban tieng Viet, giu nguyen dau."""
    s = str(s).strip()
    s = re.sub(r"https?://\S+|www\.\S+", " ", s)   # xoa URL
    s = re.sub(r"<[^>]+>", " ", s)                     # xoa HTML tag
    s = _EMOJI_RE.sub(" ", s)                           # xoa emoji
    s = re.sub(r"\s+", " ", s)                         # chuan hoa khoang trang
    return s.lower().strip()


df = pd.read_csv(DATA_PATH)
df = df.rename(columns={"Review": "text", "Label_number": "label", "Label_text": "label_text"})
df = df[["text", "label", "label_text"]].copy()
df = df.dropna(subset=["text", "label"])
df["text"]  = df["text"].astype(str).map(clean_text)
df["label"] = df["label"].astype(int)

# Loai mau qua ngan (nhieu kha nang nhieu)
df = df[df["text"].str.len() > 3].reset_index(drop=True)

print(f"Tong mau sau xu ly: {len(df):,}")
print()
print("Phan bo nhan:")
vc = df["label"].value_counts().sort_index()
for lbl, cnt in vc.items():
    print(f"  {LABEL_MAP[lbl]:12s} ({lbl}): {cnt:,}  ({cnt/len(df)*100:.1f}%)")
print()
display(df.head(5))


## 4. Phân tích Phân bố Dữ liệu (EDA)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

counts = df["label"].value_counts().sort_index()
labels_vi = [LABEL_MAP[i] for i in counts.index]

# --- Bieu do phan bo nhan ---
ax = axes[0]
colors = ["#E74C3C", "#2ECC71", "#3498DB"]
bars = ax.bar(labels_vi, counts.values, color=colors, edgecolor="white", linewidth=1.2)
ax.set_title("Phan bo nhan du lieu", fontsize=13, fontweight="bold")
ax.set_xlabel("Cam xuc")
ax.set_ylabel("So luong mau")
for bar, v in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, v + max(counts)*0.01,
            f"{v:,}", ha="center", va="bottom", fontsize=10, fontweight="bold")
ax.set_ylim(0, max(counts) * 1.12)

# --- Bieu do do dai van ban ---
ax2 = axes[1]
df["text_len"] = df["text"].str.split().str.len()
for lbl, color in zip(sorted(LABEL_MAP), colors):
    subset = df[df["label"] == lbl]["text_len"]
    ax2.hist(subset, bins=40, alpha=0.6, label=LABEL_MAP[lbl], color=color, density=True)
ax2.set_title("Phan bo do dai van ban (so tu)", fontsize=13, fontweight="bold")
ax2.set_xlabel("So tu")
ax2.set_ylabel("Mat do")
ax2.legend()
ax2.set_xlim(0, 200)

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/eda_distribution.png", dpi=120, bbox_inches="tight")
plt.show()

print(f"\nDo dai van ban (so tu):")
print(df["text_len"].describe().round(1).to_string())
df.drop(columns=["text_len"], inplace=True)


## 5. Chia Tập Dữ liệu

Tỉ lệ: **80% train / 10% val / 10% test** với stratify để giữ phân bố nhãn đều nhau.


In [ ]:
train_df, temp_df = train_test_split(
    df, test_size=0.20, random_state=SEED, stratify=df["label"]
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=SEED, stratify=temp_df["label"]
)

X_train, y_train = train_df["text"].values, train_df["label"].values
X_val,   y_val   = val_df["text"].values,   val_df["label"].values
X_test,  y_test  = test_df["text"].values,  test_df["label"].values

print(f"Train : {len(X_train):,} mau")
print(f"Val   : {len(X_val):,} mau")
print(f"Test  : {len(X_test):,} mau")

print("\nKiem tra phan bo nhan trong moi tap:")
for name, y in [("Train", y_train), ("Val", y_val), ("Test", y_test)]:
    vals, cnts = np.unique(y, return_counts=True)
    dist = " | ".join(f"{LABEL_MAP[v]}: {c:,} ({c/len(y)*100:.1f}%)" for v, c in zip(vals, cnts))
    print(f"  {name}: {dist}")


## 6. Xây dựng Pipeline

**Kiến trúc pipeline:**

```
FeatureUnion
├── TfidfVectorizer(word, ngram 1-3)   → sublinear_tf, giu dau tieng Viet
└── TfidfVectorizer(char_wb, ngram 3-5) → bat ky tu trigramm tren bien gioi tu
        ↓
SelectKBest(chi2, k=K)   → giu K dac trung co gia tri phan loai cao nhat
        ↓
LinearSVC(class_weight='balanced', C=C)
```

**Tại sao dùng FeatureUnion?**
- Word n-gram nắm ngữ nghĩa từng từ và cụm từ.
- Char n-gram nắm hình thái từ, giúp nhận dạng từ viết tắt, typo tiếng Việt.
- Kết hợp hai nguồn đặc trưng thường cho F1 tốt hơn dùng riêng lẻ.

**Tại sao dùng chi2?**
- Đo sự phụ thuộc thống kê giữa đặc trưng và nhãn.
- Loại bỏ đặc trưng nhiễu, giảm overfitting và tăng tốc độ.


In [ ]:
# --- Dinh nghia pipeline co so (chua tuning sieu tham so) ---

word_vec = TfidfVectorizer(
    analyzer="word",
    lowercase=True,
    strip_accents=None,      # Giu nguyen dau tieng Viet
    sublinear_tf=True,       # tf = log(1 + count) -> giam su chi phoi cua tu pho bien
    norm="l2",
    token_pattern=r"(?u)\b\w+\b",  # Giu ca tu 1 ky tu (tien ich, khong)
)

char_vec = TfidfVectorizer(
    analyzer="char_wb",      # char n-gram trong bien gioi tu (them khoang trang gia)
    lowercase=True,
    strip_accents=None,
    sublinear_tf=True,
    norm="l2",
)

feature_union = FeatureUnion([
    ("word", word_vec),
    ("char", char_vec),
])

pipeline = Pipeline([
    ("features", feature_union),
    ("select",   SelectKBest(chi2)),
    ("clf",      LinearSVC(
        class_weight="balanced",   # Tu dong can bang mat can bang lop
        random_state=SEED,
        max_iter=5000,
        dual="auto",               # sklearn >= 1.3: tu chon primal/dual toi uu
    )),
])

print("Pipeline da dinh nghia thanh cong:")
for name, step in pipeline.steps:
    print(f"  [{name}] {type(step).__name__}")


## 7. Tìm Siêu Tham số với RandomizedSearchCV + StratifiedKFold(5)

**Không gian tìm kiếm bao gồm:**
- `ngram_range` của word và char vectorizer
- `max_features` (giới hạn từ vựng)
- `min_df` / `max_df` (tần suất xuất hiện tối thiểu/tối đa)
- `select__k` (số lượng đặc trưng giữ lại)
- `clf__C` (hệ số regularization của SVM)

`n_iter=25` tổ hợp, `cv=StratifiedKFold(5)`, scoring = **f1_macro**.


In [ ]:
param_distributions = {
    # Word TF-IDF
    "features__word__ngram_range" : [(1, 1), (1, 2), (1, 3)],
    "features__word__max_features": [80_000, 120_000, 200_000, None],
    "features__word__min_df"       : [1, 2, 3],
    "features__word__max_df"       : [0.90, 0.95, 1.0],
    # Char TF-IDF
    "features__char__ngram_range" : [(2, 4), (3, 5), (2, 5), (3, 6)],
    "features__char__max_features": [50_000, 80_000, 120_000],
    "features__char__min_df"       : [1, 2],
    # Feature selection
    "select__k": [80_000, 150_000, 250_000, "all"],
    # SVM regularization
    "clf__C": [0.05, 0.1, 0.3, 0.5, 1.0, 2.0, 5.0, 10.0],
}

cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_distributions,
    n_iter=25,              # So to hop thu nghiem
    scoring="f1_macro",     # Tieu chi toi uu hoa
    cv=cv_strategy,
    n_jobs=-1,              # Dung toan bo CPU
    verbose=1,
    random_state=SEED,
    refit=True,             # Fit lai tren toan bo tap train voi best params
    return_train_score=True,
)

print("Bat dau RandomizedSearchCV (co the mat 10-25 phut tren CPU Colab)...")
search.fit(X_train, y_train)

print(f"\nBest CV F1-macro : {search.best_score_*100:.4f}%")
print("\nBest hyperparameters:")
for k, v in sorted(search.best_params_.items()):
    print(f"  {k}: {v}")


## 8. Phân tích Kết quả CV

In [ ]:
cv_df = pd.DataFrame(search.cv_results_).sort_values("rank_test_score")

print("Top 10 ket qua (theo F1-macro CV):")
show_cols = ["rank_test_score", "mean_test_score", "std_test_score",
             "mean_train_score", "mean_fit_time"]
display(cv_df[show_cols].head(10).round(4))

# Gap train vs val de phat hien overfitting
cv_df["gap"] = cv_df["mean_train_score"] - cv_df["mean_test_score"]
print(f"\nGap trung binh (train - val F1): {cv_df['gap'].mean():.4f}")
print(f"Gap o best params             : {cv_df['gap'].iloc[0]:.4f}")

# --- Plot top-20 CV scores ---
top_n = min(20, len(cv_df))
fig, ax = plt.subplots(figsize=(10, 5))
y_pos = np.arange(top_n)
scores = cv_df["mean_test_score"].values[:top_n][::-1]
errs   = cv_df["std_test_score"].values[:top_n][::-1]
ax.barh(y_pos, scores, xerr=errs, color="#3498DB", alpha=0.8, capsize=3)
ax.axvline(search.best_score_, color="red", linestyle="--", linewidth=1.5,
           label=f"Best = {search.best_score_:.4f}")
ax.set_yticks(y_pos)
ax.set_yticklabels([f"#{i+1}" for i in range(top_n-1, -1, -1)], fontsize=8)
ax.set_xlabel("F1-macro (CV)")
ax.set_title(f"Top {top_n} to hop sieu tham so (RandomizedSearchCV)")
ax.legend()
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/cv_scores.png", dpi=120, bbox_inches="tight")
plt.show()


## 9. Hiệu chỉnh Xác suất (Probability Calibration)

`LinearSVC` không trả về xác suất trực tiếp — nó chỉ cho điểm quyết định (decision score).

`CalibratedClassifierCV(method='sigmoid')` áp dụng Platt Scaling để biến điểm quyết định thành xác suất được hiệu chỉnh, giúp:
- F1 và độ tự tin đáng tin cậy hơn.
- Hỗ trợ phân tích mẫu có độ tự tin thấp.

Lưu ý: `cv=5` có nghĩa calibration dùng cross-val 5-fold nội bộ trên `X_train`.


In [ ]:
best_pipe = search.best_estimator_

calibrated_clf = CalibratedClassifierCV(
    best_pipe,
    method="sigmoid",   # Platt Scaling - phu hop voi SVM
    cv=5,
    n_jobs=-1,
)
calibrated_clf.fit(X_train, y_train)

print("Calibration hoan thanh.")

# Kiem tra nhanh tren val
val_pred = calibrated_clf.predict(X_val)
val_acc  = accuracy_score(y_val, val_pred)
_, _, val_f1, _ = precision_recall_fscore_support(y_val, val_pred, average="macro", zero_division=0)
print(f"Val Accuracy  : {val_acc*100:.4f}%")
print(f"Val F1-macro  : {val_f1*100:.4f}%")


## 10. Đánh giá Toàn diện trên Tập Test

Tập test **chưa bao giờ được dùng** trong quá trình huấn luyện hay chỉnh tham số.  
Đây là số liệu phản ánh hiệu năng thực tế của mô hình.


In [ ]:
# --- Du doan tren tap test ---
y_pred = calibrated_clf.predict(X_test)
y_prob = calibrated_clf.predict_proba(X_test)

# --- Cac chi so chinh ---
acc           = accuracy_score(y_test, y_pred)
p_mac, r_mac, f1_mac, _ = precision_recall_fscore_support(
    y_test, y_pred, average="macro", zero_division=0)
p_wt,  r_wt,  f1_wt,  _ = precision_recall_fscore_support(
    y_test, y_pred, average="weighted", zero_division=0)

print("=" * 55)
print("  KET QUA DANH GIA TREN TAP TEST")
print("=" * 55)
print(f"  Accuracy          : {acc*100:.4f}%")
print(f"  Precision (macro) : {p_mac*100:.4f}%")
print(f"  Recall    (macro) : {r_mac*100:.4f}%")
print(f"  F1        (macro) : {f1_mac*100:.4f}%")
print("-" * 55)
print(f"  Precision (weighted): {p_wt*100:.4f}%")
print(f"  Recall    (weighted): {r_wt*100:.4f}%")
print(f"  F1        (weighted): {f1_wt*100:.4f}%")
print("=" * 55)

print("\nBao cao phan loai chi tiet:")
print(classification_report(
    y_test, y_pred,
    target_names=CLASS_NAMES,
    digits=4,
    zero_division=0,
))


## 11. Ma trận Nhầm lẫn và Phân bố Độ Tự tin

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- Ma tran nham lan ---
cm = confusion_matrix(y_test, y_pred)
ax = axes[0]
if HAS_SEABORN:
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="YlGnBu",
        xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
        linewidths=0.5, ax=ax,
    )
else:
    im = ax.imshow(cm, cmap="YlGnBu")
    fig.colorbar(im, ax=ax)
    ax.set_xticks([0, 1, 2]); ax.set_xticklabels(CLASS_NAMES)
    ax.set_yticks([0, 1, 2]); ax.set_yticklabels(CLASS_NAMES)
    for i in range(3):
        for j in range(3):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=12)

ax.set_title("Ma tran nham lan\nTF-IDF + LinearSVC (Calibrated)", fontsize=12, fontweight="bold")
ax.set_xlabel("Nhan du doan")
ax.set_ylabel("Nhan thuc te")

# --- Phan bo do tu tin ---
conf    = y_prob.max(axis=1)
correct = (y_pred == y_test)
ax2 = axes[1]
if HAS_SEABORN:
    sns.kdeplot(conf[correct],  fill=True, alpha=0.6, label="Du doan dung", ax=ax2, color="#2ECC71")
    sns.kdeplot(conf[~correct], fill=True, alpha=0.6, label="Du doan sai",  ax=ax2, color="#E74C3C")
else:
    ax2.hist(conf[correct],  bins=30, alpha=0.6, density=True, label="Du doan dung", color="#2ECC71")
    ax2.hist(conf[~correct], bins=30, alpha=0.6, density=True, label="Du doan sai",  color="#E74C3C")

ax2.axvline(0.5, color="gray", linestyle="--", linewidth=1)
ax2.set_title("Phan bo do tu tin cua mo hinh", fontsize=12, fontweight="bold")
ax2.set_xlabel("Xac suat du doan cao nhat")
ax2.set_ylabel("Mat do")
ax2.legend()

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/confusion_matrix_confidence.png", dpi=120, bbox_inches="tight")
plt.show()

# --- Phan bo nham lan theo tung lop ---
print("\nChi tiet nham lan theo tung lop:")
for i, true_name in enumerate(CLASS_NAMES):
    row_sum = cm[i].sum()
    errs = [(CLASS_NAMES[j], cm[i, j]) for j in range(3) if j != i and cm[i, j] > 0]
    errs_str = " | ".join(f"bi nham thanh {n}: {c} ({c/row_sum*100:.1f}%)" for n, c in errs)
    print(f"  {true_name}: {errs_str if errs else 'khong co nham lan nao dang ke'}")


## 12. Biểu đồ Metrics theo từng Lớp

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

p_per, r_per, f1_per, sup_per = precision_recall_fscore_support(
    y_test, y_pred, average=None, labels=[0, 1, 2], zero_division=0
)

x = np.arange(3)
width = 0.25

fig, ax = plt.subplots(figsize=(10, 5))
b1 = ax.bar(x - width,     p_per  * 100, width, label="Precision", color="#3498DB", alpha=0.85)
b2 = ax.bar(x,             r_per  * 100, width, label="Recall",    color="#2ECC71", alpha=0.85)
b3 = ax.bar(x + width,     f1_per * 100, width, label="F1-score",  color="#E74C3C", alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES, fontsize=11)
ax.set_ylabel("Gia tri (%)")
ax.set_ylim(0, 115)
ax.set_title("Precision / Recall / F1 theo tung lop (Tap Test)", fontsize=13, fontweight="bold")
ax.legend(loc="upper right")
ax.axhline(y=90, color="gray", linestyle="--", linewidth=0.8, alpha=0.7)

for bars in [b1, b2, b3]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, h + 0.5,
                f"{h:.1f}", ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/per_class_metrics.png", dpi=120, bbox_inches="tight")
plt.show()


## 13. Demo Dự đoán Cảm xúc

In [ ]:
def predict_sentiment(text: str) -> dict:
    """Du doan cam xuc cho mot doan van ban tieng Viet."""
    text_clean = _EMOJI_RE.sub(" ", re.sub(r"\s+", " ", str(text).lower().strip()))
    pred_label = int(calibrated_clf.predict([text_clean])[0])
    proba      = calibrated_clf.predict_proba([text_clean])[0]
    return {
        "text"      : text,
        "label"     : LABEL_MAP[pred_label],
        "confidence": float(proba.max()),
        "proba"     : {LABEL_MAP[i]: round(float(p), 4) for i, p in enumerate(proba)},
    }


test_samples = [
    "san pham qua te, moi dung da hong ngay",
    "giao hang nhanh, dong goi tot, hai long",
    "chat luong tuyet voi, se ung ho tiep",
    "5 sao cho vui chu khong bao gio mua lai",
    "binh thuong, khong co gi dac biet",
    "shop tu van nhiet tinh, giao hang dung han, san pham chinh hang 100%",
    "hang bi loi, lien he shop khong phan hoi, rat that vong",
]

print(f"{'Van ban':<55} {'Du doan':<12} {'Tu tin':>8}")
print("-" * 80)
for s in test_samples:
    result = predict_sentiment(s)
    label  = result["label"]
    conf   = result["confidence"]
    trunc  = (s[:52] + "...") if len(s) > 55 else s
    print(f"{trunc:<55} {label:<12} {conf*100:>7.2f}%")

print()
print("Chi tiet xac suat cua mau cuoi:")
r = predict_sentiment(test_samples[-1])
for lbl, p in r["proba"].items():
    print(f"  {lbl}: {p:.4f}")


## 14. Lưu Model và Metrics

In [ ]:
# Luu model da calibrate (bao gom ca vectorizer ben trong pipeline)
model_path   = f"{OUT_DIR}/tfidf_svm_calibrated.joblib"
metrics_path = f"{OUT_DIR}/metrics.json"

joblib.dump(calibrated_clf, model_path, compress=3)

metrics = {
    "model"                  : "TF-IDF(word+char) + SelectKBest(chi2) + LinearSVC + CalibratedClassifierCV",
    "best_cv_f1_macro"       : round(float(search.best_score_), 6),
    "test_accuracy"          : round(float(acc), 6),
    "test_precision_macro"   : round(float(p_mac), 6),
    "test_recall_macro"      : round(float(r_mac), 6),
    "test_f1_macro"          : round(float(f1_mac), 6),
    "test_precision_weighted": round(float(p_wt), 6),
    "test_recall_weighted"   : round(float(r_wt), 6),
    "test_f1_weighted"       : round(float(f1_wt), 6),
    "best_params"            : search.best_params_,
    "data_split"             : {
        "train": int(len(X_train)),
        "val"  : int(len(X_val)),
        "test" : int(len(X_test)),
    },
    "seed": SEED,
}

with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2, default=str)

print("Da luu cac file vao:", OUT_DIR)
print(f"  Model : {model_path}")
print(f"  Metrics: {metrics_path}")
print()
print("Tom tat ket qua cuoi:")
print(json.dumps({k: v for k, v in metrics.items() if "test" in k or "cv" in k},
                 ensure_ascii=False, indent=2))


## 15. Kiểm tra Tải lại Model (Load & Verify)

Bước này xác nhận model đã lưu có thể được tải và dự đoán chính xác — quan trọng cho deployment.


In [ ]:
# Tai lai model tu file
loaded_clf = joblib.load(model_path)

# Kiem tra ket qua phai giong he voi calibrated_clf
y_pred_loaded = loaded_clf.predict(X_test)
assert np.array_equal(y_pred, y_pred_loaded), "Ket qua du doan khong khop! Luu model that bai."

acc_loaded = accuracy_score(y_test, y_pred_loaded)
print(f"Accuracy model tai lai: {acc_loaded*100:.4f}%  (phai bang {acc*100:.4f}%)")
print("Kiem tra load model: THANH CONG")
